# SQL Analysis: Makeup Products — Price, Quality & Green Claims
## Lisa, Eliram, and Judi
### Tool: SQLite via Python (sqlite3 + pandas)

All queries use our primary dataset from the Makeup API, supplemented by a
manually created `green_terms` reference table. Queries are organized around
three analytical themes:
1. **Data quality checks** (Queries 1–2)
2. **Price & market structure** (Queries 3, 7–10)
3. **Green claims analysis** (Queries 4–6, 11–13)

In [1]:
# Here we are importing the necessary libraries/ packages for our analysis. 
import sqlite3
import pandas as pd
import numpy as np
import requests

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 1200)

# We are setting the max columns and width displayed for optimal memory usage.


### Pull data from the same API as the main notebook 

In [3]:
BASE_URL = "http://makeup-api.herokuapp.com/api/v1/products.json"
resp = requests.get(BASE_URL, timeout=30)
resp.raise_for_status()
data_json = resp.json()
df_raw = pd.DataFrame(data_json)

### Pre-process columns before loading into SQLite

In [4]:

df_raw["price"]              = pd.to_numeric(df_raw["price"], errors="coerce")
df_raw["rating"]             = pd.to_numeric(df_raw["rating"], errors="coerce")
df_raw["description_length"] = df_raw["description"].fillna("").str.len()
df_raw["word_count"]         = df_raw["description"].fillna("").str.split().str.len()
# Flatten the tag_list (Python list) into a comma-separated lowercase string
df_raw["tags"]               = df_raw["tag_list"].apply(
    lambda x: ",".join(x).lower() if isinstance(x, list) else ""
)

print(f"Records loaded: {len(df_raw)}")

Records loaded: 931


### Create in-memory SQLite database 

In [5]:

conn = sqlite3.connect(":memory:")


### TABLE 1: products  (main dataset from the API)

In [6]:
# We decided to grab the following column names for our SQL analysis
cols_for_sql = [
    "id", "brand", "name", "price", "rating",
    "product_type", "category", "description",
    "description_length", "word_count", "tags"
]
df_raw[cols_for_sql].to_sql("products", conn, index=False, if_exists="replace")

931

### TABLE 2: green_terms  (hand-coded reference / lookup table)
###   Contains marketing terms associated with "clean" or "natural" beauty

In [7]:

# We are defining a new variable to hold our data frame of the most popular terms + categories we found to 
# describe eco friendly beauty 
green_terms_df = pd.DataFrame({
    "term": [
        "natural", "organic", "clean", "vegan", "cruelty free",
        "chemical free", "eco", "mineral", "plant", "sustainable",
        "non-toxic", "gluten free"
    ],
    "category": [
        "nature-based", "certified", "clean beauty", "ethical", "ethical",
        "clean beauty", "eco-friendly", "nature-based", "nature-based",
        "eco-friendly", "clean beauty", "dietary"
    ]
})
green_terms_df.to_sql("green_terms", conn, index=False, if_exists="replace")
# We put this data frame into SQL and then check how many rows we have for products and green terms. 

for tbl in ["products", "green_terms"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tbl}", conn).iloc[0, 0]
    print(f"  {tbl}: {n} rows")

  products: 931 rows
  green_terms: 12 rows


##  QUERY 1: Basic Dataset Overview 
### What:  High-level summary statistics for the full products table.
### Why:   First step before any analysis — understand scale, price/rating ranges,
###        and extent of missing data.
### How:   Aggregation with COUNT, AVG, MIN, MAX; CASE WHEN to handle NULLs/zeros.

In [8]:
q1 = """
SELECT
    COUNT(*)                                                              AS total_products,
    COUNT(DISTINCT brand)                                                 AS unique_brands,
    COUNT(DISTINCT product_type)                                          AS unique_product_types,
    ROUND(AVG(CASE WHEN price > 0 THEN price END), 2)                    AS avg_price,
    ROUND(MIN(CASE WHEN price > 0 THEN price END), 2)                    AS min_valid_price,
    ROUND(MAX(price), 2)                                                  AS max_price,
    SUM(CASE WHEN price IS NULL OR price = 0 THEN 1 ELSE 0 END)          AS invalid_price_count,
    ROUND(AVG(rating), 2)                                                 AS avg_rating,
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END)                       AS missing_rating_count,
    ROUND(100.0 * SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END)
          / COUNT(*), 1)                                                  AS rating_missing_pct
FROM products;
"""
#Here, we are pulling the data from our API and making it into a more readable format. 
#For example, we go the summary statistics and renamed our columns using the AS operator.

print("=== Query 1: Basic Dataset Overview ===")
pd.read_sql(q1, conn)

# We print it to see our output


=== Query 1: Basic Dataset Overview ===


,total_products,unique_brands,unique_product_types,avg_price,min_valid_price,max_price,invalid_price_count,avg_rating,missing_rating_count,rating_missing_pct
0,931,57,10,17.26,1.99,77.0,54,4.32,591,63.5


### Output: 931 products, ~50 brands, 9 product types.
### 63.5% of products are missing ratings — a significant limitation noted in EDA.


## QUERY 2: Missing Data Analysis by Column 
### What:  Count and percentage of missing values for each key column.
### Why:   Systematic data quality check. Informs which analyses are reliable.
### How:   CASE WHEN to flag NULLs per column; UNION ALL to stack results vertically.

In [9]:
# Here we are using the SELECT operator for each of our categories from the products data frame. We are calculating
# the total missing products using the SUM operator. We then calculate a total percentage of missing data from
# each column/ category for our reference, and round that number. 

q2 = """
SELECT 'brand'        AS column_name,
    SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END)                       AS missing_count,
    ROUND(100.0 * SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END)
          / COUNT(*), 1)                                                  AS missing_pct
FROM products
UNION ALL
SELECT 'price',
    SUM(CASE WHEN price IS NULL OR price = 0 THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN price IS NULL OR price = 0 THEN 1 ELSE 0 END)
          / COUNT(*), 1)
FROM products
UNION ALL
SELECT 'rating',
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END)
          / COUNT(*), 1)
FROM products
UNION ALL
SELECT 'description',
    SUM(CASE WHEN description IS NULL OR description = '' THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN description IS NULL OR description = '' THEN 1 ELSE 0 END)
          / COUNT(*), 1)
FROM products
UNION ALL
SELECT 'product_type',
    SUM(CASE WHEN product_type IS NULL THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN product_type IS NULL THEN 1 ELSE 0 END)
          / COUNT(*), 1)
FROM products
ORDER BY missing_pct DESC;
"""
# This helps us have a better idea of what data is missing so that we can address it later on. 

print("=== Query 2: Missing Data by Column ===")
pd.read_sql(q2, conn)


=== Query 2: Missing Data by Column ===


,column_name,missing_count,missing_pct
0,rating,591,63.5
1,price,54,5.8
2,description,25,2.7
3,brand,12,1.3
4,product_type,0,0.0


## Output: 
rating (63.5%) and price_sign (60.5%) have the most missing data.
price itself is mostly complete (1.5% missing/zero), so price analysis is viable.


In [10]:
# ── QUERY 3: Price Tier Distribution ─────────────────────────────────────────
# What:  Classify products into 4 price tiers; compute product count, avg price,
#        avg rating, and brand diversity per tier.
# Why:   Establishes price segmentation framework used throughout the analysis.
#        Lets us test whether higher-tier products also get better ratings.
# How:   CASE WHEN creates tier labels; GROUP BY aggregates metrics per tier.

q3 = """
SELECT
    CASE
        WHEN price < 10  THEN '1_Budget      (<$10)'
        WHEN price < 20  THEN '2_Mid-range   ($10–19)'
        WHEN price < 35  THEN '3_Premium     ($20–34)'
        ELSE                  '4_Luxury      ($35+)'
    END                                AS price_tier,
    COUNT(*)                           AS product_count,
    ROUND(AVG(price), 2)               AS avg_price,
    ROUND(MIN(price), 2)               AS min_price,
    ROUND(MAX(price), 2)               AS max_price,
    ROUND(AVG(rating), 2)              AS avg_rating,
    COUNT(DISTINCT brand)              AS distinct_brands
FROM products
WHERE price > 0
GROUP BY price_tier
ORDER BY price_tier;
"""

print("=== Query 3: Price Tier Distribution (GROUP BY) ===")
pd.read_sql(q3, conn)



=== Query 3: Price Tier Distribution (GROUP BY) ===


,price_tier,product_count,avg_price,min_price,max_price,avg_rating,distinct_brands
0,1_Budget (<$10),259,7.33,1.99,9.99,4.31,23
1,2_Mid-range ($10–19),321,14.18,10.00,19.99,4.29,28
2,3_Premium ($20–34),232,25.33,20.00,34.50,4.43,22
3,4_Luxury ($35+),65,43.29,35.00,77.00,4.33,10


### Output: 
Most products fall in the Mid-range tier ($10–19).
Avg ratings are similar across tiers (~4.3), supporting weak price-quality link.


## QUERY 4: Green Tag Distribution by Product Type 
### What:  Count how many products per product_type carry at least one green/ethical
### self-declared tag (vegan, organic, natural, chemical free).
### Why:   Baseline for the green hypothesis — which categories skew "green"?
### How:   CASE WHEN + LIKE to detect green tags; GROUP BY product_type.

In [11]:
# We SELECT product type and count all of its chosen categories (for example, for the green tags, when a description contains
# any of the green tokens ("%vegan%","%organic%", etc.) we can count them as 1 (TRUE) or 0 (FALSE) if not present and put them
# in a column using AS from our products table, taking out NULLS with WHERE __ IS NOT NULL. Grouping by product_type
# And Ordering them as Descending based on percentage green

q4 = """
SELECT
    product_type,
    COUNT(*)                                                                  AS total_products,
    SUM(CASE WHEN tags LIKE '%vegan%'        THEN 1 ELSE 0 END)              AS vegan_count,
    SUM(CASE WHEN tags LIKE '%organic%'      THEN 1 ELSE 0 END)              AS organic_count,
    SUM(CASE WHEN tags LIKE '%cruelty%'      THEN 1 ELSE 0 END)              AS cruelty_free_count,
    SUM(CASE WHEN tags LIKE '%natural%'      THEN 1 ELSE 0 END)              AS natural_count,
    SUM(CASE WHEN tags LIKE '%vegan%'
              OR tags LIKE '%organic%'
              OR tags LIKE '%natural%'
              OR tags LIKE '%chemical free%' THEN 1 ELSE 0 END)              AS any_green_tag,
    ROUND(
        100.0 * SUM(CASE WHEN tags LIKE '%vegan%'
                          OR tags LIKE '%organic%'
                          OR tags LIKE '%natural%'
                          OR tags LIKE '%chemical free%' THEN 1 ELSE 0 END)
        / COUNT(*), 1)                                                        AS pct_green
FROM products
WHERE product_type IS NOT NULL
GROUP BY product_type
ORDER BY pct_green DESC;
"""

print("=== Query 4: Green Tag Distribution by Product Type (GROUP BY) ===")
pd.read_sql(q4, conn)



=== Query 4: Green Tag Distribution by Product Type (GROUP BY) ===


,product_type,total_products,vegan_count,organic_count,cruelty_free_count,natural_count,any_green_tag,pct_green
0,nail_polish,60,8,0,0,14,17,28.3
1,eyeshadow,86,13,7,0,9,21,24.4
2,bronzer,69,7,1,0,9,14,20.3
3,mascara,92,6,2,0,13,17,18.5
4,blush,78,5,1,0,8,11,14.1
5,lip_liner,29,3,0,1,2,4,13.8
6,foundation,166,8,0,3,10,16,9.6
7,eyeliner,148,9,1,0,10,14,9.5
8,lipstick,154,8,2,2,7,14,9.1
9,eyebrow,49,0,0,0,0,0,0.0


## Output:
Lip products and foundations tend to have the highest % of green tags.

##  QUERY 5: Match Products to Green Terms Reference Table 
### What:  Find products whose NAME or TAGS contain any term from the green_terms
###        lookup table; summarize count, avg price, and avg rating per term.
### Why:   More systematic than Query 4 — captures green language even when not
###        explicitly tagged. Tests whether certain green keywords co-occur with
###        price premiums.
### How:   JOIN on LIKE condition between products and green_terms reference table.


In [12]:
# This is our first join (Join #1). We SELECT and rename with AS our green-terms categories, product type, product id DISTINC COUNT
# Then the rounded (two decimals) avergae price of products and average rating.
# We join our green_terms categories with our products data table based on lower-case standardized tokens from our green categories
# based on product name and product tags. Then, we filter by product prices being larger than 0 (not free) 
#and group by green category, green term, and product type to then order by the products matched to green term category (Descending)
#Feature Engineered matched_prodcts, avg_price, and avg_rating
q5 = """
SELECT
    gt.category                        AS green_category,
    gt.term                            AS matched_term,
    p.product_type,
    COUNT(DISTINCT p.id)               AS matched_products,
    ROUND(AVG(p.price), 2)             AS avg_price,
    ROUND(AVG(p.rating), 2)            AS avg_rating
FROM products p
JOIN green_terms gt
    ON LOWER(p.name)  LIKE '%' || gt.term || '%'
    OR LOWER(p.tags)  LIKE '%' || gt.term || '%'
WHERE p.price > 0
GROUP BY gt.category, gt.term, p.product_type
ORDER BY matched_products DESC, gt.category;
"""

print("=== Query 5: Products Matched to Green Terms (JOIN #1) ===")
pd.read_sql(q5, conn)



=== Query 5: Products Matched to Green Terms (JOIN #1) ===


,green_category,matched_term,product_type,matched_products,avg_price,avg_rating
0,dietary,gluten free,eyeliner,17,15.53,4.49
1,nature-based,natural,mascara,14,19.17,3.96
2,nature-based,natural,nail_polish,14,15.06,4.09
3,dietary,gluten free,blush,12,19.95,4.46
4,dietary,gluten free,foundation,12,19.67,4.19
5,ethical,vegan,eyeshadow,12,15.28,4.51
6,nature-based,mineral,foundation,12,20.37,4.35
7,nature-based,natural,foundation,12,26.20,4.08
8,dietary,gluten free,mascara,10,16.99,3.98
9,nature-based,natural,eyeliner,10,17.02,3.91


## Output: 
'vegan' and 'cruelty free' are the most common green terms across all types.
'organic' and 'natural' appear less frequently — may represent premium niche.

QUERY 6: Green-Labeled vs Non-Green Price & Rating Comparison

What: For each product_type, compare avg price and avg rating between

green-labeled and non-green products.

Why: Core hypothesis test — do green marketing claims correlate with a price

premium? And does that premium come with better or worse ratings?

How: CTE (style 1 subquery) flags each product as green/non-green using EXISTS;

outer query groups and aggregates. The CTE result is then JOINed implicitly

via GROUP BY (JOIN #2 is the brand_stats join in Q13).

In [1]:
# We use WITH to define a temporary table named green_flag, for future use.
#Then, we SELECT product id, type, price, and rating, as well as the conditional subquery
#FROM green_terms, filtering using WHERE to convert product name and tags to lower case and
#then check if any "green term" exists anywhere inside those strings. Then, the product is
#labeled "Green-label" or "Non-green".
#Finally, the source is FROM products and we filter for free or erroneous prices.
#Then, the second part of the query uses this green_flag table, counting the total number
#of products that fall into each category. Then, we round AVG price and AVG rating for standard.
# We run these on green_flag temporary table. To, then, Group by product type and green label
#and order by product type and green_label

q6 = """
WITH green_flag AS (
    SELECT
        p.id,
        p.product_type,
        p.price,
        p.rating,
        CASE
            WHEN EXISTS (
                SELECT 1
                FROM green_terms gt
                WHERE LOWER(p.name) LIKE '%' || gt.term || '%'
                   OR LOWER(p.tags) LIKE '%' || gt.term || '%'
            ) THEN 'Green-Labeled'
            ELSE 'Non-Green'
        END AS green_label
    FROM products p
    WHERE p.price > 0
)
SELECT
    product_type,
    green_label,
    COUNT(*)               AS product_count,
    ROUND(AVG(price), 2)   AS avg_price,
    ROUND(AVG(rating), 2)  AS avg_rating
FROM green_flag
GROUP BY product_type, green_label
ORDER BY product_type, green_label;
"""
print("=== Query 6: Green vs Non-Green Comparison (CTE subquery) ===")
df6 = pd.read_sql(q6, conn)
df6

=== Query 6: Green vs Non-Green Comparison (CTE subquery) ===


NameError: name 'pd' is not defined

## Output: 
Green-labeled products tend to have slightly higher avg prices in most
categories, but rating differences are small — green claims don't clearly
translate to better consumer satisfaction.


## QUERY 7: Brand Performance Summary 
### What:  For brands with ≥5 products, aggregate price range, avg rating, number of
###        categories covered, and count of green-tagged products.
### Why:   Understand brand-level positioning. Brands with broad category coverage
###        and high ratings may be better bets for value-conscious consumers.
### How:   GROUP BY brand; HAVING filters for statistical meaningfulness (≥5 products).


In [14]:
# Here we make the variabkle q7 and assign the following queries:
q7 = """
SELECT
    brand,
    COUNT(*)                                                              AS total_products,
    COUNT(DISTINCT product_type)                                          AS categories_covered,
    ROUND(AVG(price), 2)                                                  AS avg_price,
    ROUND(MIN(price), 2)                                                  AS min_price,
    ROUND(MAX(price), 2)                                                  AS max_price,
    ROUND(AVG(rating), 2)                                                 AS avg_rating,
    SUM(CASE WHEN rating IS NOT NULL  THEN 1 ELSE 0 END)                  AS rated_products,
    SUM(CASE WHEN tags LIKE '%vegan%'
              OR tags LIKE '%organic%'
              OR tags LIKE '%natural%' THEN 1 ELSE 0 END)                 AS green_products
FROM products
WHERE price > 0
  AND brand IS NOT NULL
GROUP BY brand
HAVING COUNT(*) >= 5
ORDER BY avg_rating DESC, total_products DESC
LIMIT 20;
"""
# We first start with the SELECT operator where we grab the brand and create the following columns:
# total_ products ---> We grab the total amount using the COUNT operator
# categories_covered ---> We grab the total count of distinct product types witht the DISTINCT operator
# avg_price & avg_rating ---> We are rounding the average price & rating to two decimal points with the ROUND & AVG operators
# min_price & max_price ---> We are rounding the max price using the MAX operator

# In order to count the number of not-null items, we have to use the SUM operator and assign the values 1 if not null, 0 if otherwise.
# For our pattern matching/ tokenization we use the same structure with the SUM operator, grouping the use of the words "vegan", "organic", and "natural" as 
# a new column called green_products

# Lastly, from the products table, where price is greater than zero and the brand name is present we are grouping by the following atributes:
#  -- brand count have greater than or equal to 5
#  -- ordering from the highest average rating, and the lowest total product counts
#  -- limiting the coutnt to 20




print("=== Query 7: Brand Performance Summary (GROUP BY + HAVING) ===")
pd.read_sql(q7, conn) 

# Here we read and print our sql queries to derive the following table: 

=== Query 7: Brand Performance Summary (GROUP BY + HAVING) ===


,brand,total_products,categories_covered,avg_price,min_price,max_price,avg_rating,rated_products,green_products
0,annabelle,11,5,9.81,6.99,11.99,4.97,6,0
1,cargo cosmetics,20,7,29.25,19.00,42.00,4.85,11,0
2,dr. hauschka,12,6,33.92,22.00,45.00,4.67,10,7
3,wet n wild,12,5,4.31,1.99,6.99,4.67,9,0
4,milani,13,7,9.07,3.99,12.99,4.48,10,1
5,pacifica,13,8,25.46,14.00,60.00,4.47,12,13
6,revlon,29,7,13.49,6.99,19.99,4.46,23,0
7,pure anada,16,8,14.25,8.00,25.99,4.46,15,16
8,covergirl,54,9,9.68,4.49,15.99,4.43,33,0
9,e.l.f.,27,8,6.77,3.99,13.99,4.38,22,21


## Output: 
Top-rated brands tend to be mid-size (5–20 products). Brands with more
green products don't consistently have higher ratings.

## QUERY 8: Price Rank within Product Type 
### What:  Rank each product by price within its product_type, and show how far
###        each product's price deviates from its category average.
### Why:   Lets consumers see where a specific product sits price-wise among peers —
###        more informative than raw price alone.
### How:   RANK() OVER (PARTITION BY product_type ORDER BY price DESC) assigns rank;
###        AVG() OVER (PARTITION BY product_type) computes the in-category mean.
###       Both are window functions — no GROUP BY, no row collapsing.


In [15]:
# Here we assign the following query to the variable "q8"
q8 = """
SELECT
    brand,
    name,
    product_type,
    price,
    rating,
    RANK()  OVER (PARTITION BY product_type ORDER BY price DESC)          AS price_rank,
    COUNT(*) OVER (PARTITION BY product_type)                              AS products_in_type,
    ROUND(AVG(price) OVER (PARTITION BY product_type), 2)                 AS type_avg_price,
    ROUND(price - AVG(price) OVER (PARTITION BY product_type), 2)         AS price_vs_type_avg
FROM products
WHERE price > 0
ORDER BY product_type, price_rank;
"""
# We are selecting the brand, name, product, price, and rating. Then, using the window functions RANK() and PARTITION we are ranking based on
# how far the product price deviates from the average category product price. We assign this value to the column "price_rank". We continue to 
# partition by (do calculations across rows, keeping the rows) to list our average prices based on product types and get the difference 
# between the price of the product and the average price for the product type. 

# Lastly, we are specifiying that we are selecting this from the products table, where price is greater than 0 dollars, and ordering by product 
# type and the price rank


print("=== Query 8: Price Rank within Product Type (Window Function #1) ===")
pd.read_sql(q8, conn)

# We read this query 


=== Query 8: Price Rank within Product Type (Window Function #1) ===


,brand,name,product_type,price,rating,price_rank,products_in_type,type_avg_price,price_vs_type_avg
0,stila,Stila Convertible Colour Palette Sinrise Splendor,blush,51.00,NaN,1,72,18.11,32.89
1,stila,Stila Convertible Colour Palette Sunset Serenade,blush,51.00,NaN,1,72,18.11,32.89
2,smashbox,L.A. Lights Palette,blush,35.00,NaN,3,72,18.11,16.89
3,clinique,Sculptionary&trade; Cheek Contouring Palette,blush,34.00,NaN,4,72,18.11,15.89
4,dior,Diorblush Sculpt,blush,32.50,NaN,5,72,18.11,14.39
...,...,...,...,...,...,...,...,...,...
872,maybelline,Maybelline Color Show Nail Lacquer Jewels,nail_polish,4.49,3.0,55,59,13.12,-8.63
873,maybelline,Maybelline Color Show Nail Lacquer Veils,nail_polish,4.49,4.0,55,59,13.12,-8.63
874,maybelline,Maybelline Color Show Nail Lacquer,nail_polish,4.49,3.3,55,59,13.12,-8.63
875,sinful colours,Sinful Colours Nail Polish,nail_polish,2.99,4.3,58,59,13.12,-10.13


## Output: 
Reveals which brands dominate the high-price end of each category.
price_vs_type_avg shows how far above/below a product sits from its category norm.

## QUERY 9: Rating Percentile within Product Type 
### What:  Determine and compare individual rating to category average.
### Why:   Identifies top performers relative to their category peers — useful for
###        building a "recommended" flag (which we also did in Python EDA).
### How:   PERCENT_RANK() OVER (PARTITION BY product_type ORDER BY rating):
###        returns 0 for the lowest-rated, 1.0 for the highest-rated in each group.

In [16]:
# Here, we SELECT brand, name, product type, price, rating, as well as the percentage rank ordered by rating by product type. Also, the average rating for each
# make up type and the rating vs type average difference by product type to see how different the product is from its type's average rating.
# We filter by non-null ratings using WHERE and by price being larger than 0.
#Then, we order by product type and rating percentage rank, in descending order.

q9 = """
SELECT
    brand,
    name,
    product_type,
    price,
    rating,
    ROUND(PERCENT_RANK() OVER (PARTITION BY product_type ORDER BY rating), 2)  AS rating_pct_rank,
    ROUND(AVG(rating)    OVER (PARTITION BY product_type), 2)                  AS type_avg_rating,
    ROUND(rating - AVG(rating) OVER (PARTITION BY product_type), 2)            AS rating_vs_type_avg
FROM products
WHERE rating IS NOT NULL
  AND price  > 0
ORDER BY product_type, rating_pct_rank DESC;
"""

print("=== Query 9: Rating Percentile within Product Type (Window Function #2) ===")
pd.read_sql(q9, conn)



=== Query 9: Rating Percentile within Product Type (Window Function #2) ===


,brand,name,product_type,price,rating,rating_pct_rank,type_avg_rating,rating_vs_type_avg
0,NaN,Fake Bake Blush Legal Sunburn,blush,15.99,5.0,0.69,4.42,0.58
1,covergirl,CoverGirl truBLEND Blush in Medium Rose,blush,13.99,5.0,0.69,4.42,0.58
2,cargo cosmetics,Cargo Cosmetics Swimmables Water Resistant Blush,blush,29.00,5.0,0.69,4.42,0.58
3,mineral fusion,Mineral Fusion Blush,blush,32.00,5.0,0.69,4.42,0.58
4,covergirl,CoverGirl Clean Glow Blush,blush,8.99,5.0,0.69,4.42,0.58
...,...,...,...,...,...,...,...,...
335,orly,Orly EPIX Flexible Color,nail_polish,13.49,3.0,0.06,4.12,-1.12
336,orly,Orly Nail Lacquer,nail_polish,8.00,3.0,0.06,4.12,-1.12
337,maybelline,Maybelline Color Show Nail Lacquer Jewels,nail_polish,4.49,3.0,0.06,4.12,-1.12
338,moov,Moov Cosmetics Caribbean Wedding Collection,nail_polish,14.99,2.0,0.03,4.12,-2.12


## Output: 
Products with rating_pct_rank = 1.0 are the top-rated in their category.
Combining price_rank (Q8) with rating_pct_rank gives a value-for-money signal.

## QUERY 10: Products Priced Above Their Category Average 
### What:  Select only products whose price exceeds the average for their product_type,
###        and show how large the premium is.
### Why:   Isolates "premium" products within each category for targeted analysis —
###        useful for testing if premium positioning correlates with better ratings.
### How:   Correlated subquery in WHERE clause: for each row p, the inner SELECT
###        re-computes AVG(price) filtered to p's product_type. This is style 1
###        subquery (inline, correlated).

In [17]:
# Here, we SELECT product brand, name, product_type, product_price, and product_rating as well as the rounded subquery of average of p2
# p2 is the new table for product type form the product table, filtered by price larger than 0.
#using this p2, we also selected the average of p2 price for price >0 and the rounded difference of product price - average of p2 price, renamed AS
# category_avg_price and price_premium, respectively.
#Then, we filter using WHERe for product price being larger than p3 price average (a subquery where p3.prudct type is p.product type filtered to only include positive prices
#also filtering, finally, for product price larger than 0. ORDERED BY product-type and price_premium, descending.

q10 = """
SELECT
    p.brand,
    p.name,
    p.product_type,
    p.price,
    p.rating,
    ROUND(
        (SELECT AVG(p2.price)
         FROM products p2
         WHERE p2.product_type = p.product_type
           AND p2.price > 0), 2)                                           AS category_avg_price,
    ROUND(
        p.price - (SELECT AVG(p2.price)
                   FROM products p2
                   WHERE p2.product_type = p.product_type
                     AND p2.price > 0), 2)                                 AS price_premium
FROM products p
WHERE p.price > (
    SELECT AVG(p3.price)
    FROM products p3
    WHERE p3.product_type = p.product_type
      AND p3.price > 0
)
AND p.price > 0
ORDER BY p.product_type, price_premium DESC;
"""

print("=== Query 10: Products Above Category Average Price (Correlated Subquery) ===")
pd.read_sql(q10, conn)



=== Query 10: Products Above Category Average Price (Correlated Subquery) ===


,brand,name,product_type,price,rating,category_avg_price,price_premium
0,stila,Stila Convertible Colour Palette Sinrise Splendor,blush,51.00,NaN,18.11,32.89
1,stila,Stila Convertible Colour Palette Sunset Serenade,blush,51.00,NaN,18.11,32.89
2,smashbox,L.A. Lights Palette,blush,35.00,NaN,18.11,16.89
3,clinique,Sculptionary&trade; Cheek Contouring Palette,blush,34.00,NaN,18.11,15.89
4,dior,Diorblush Sculpt,blush,32.50,NaN,18.11,14.39
...,...,...,...,...,...,...,...
363,moov,Moov Cosmetics St. Tropez Collection,nail_polish,14.99,NaN,13.12,1.87
364,moov,Moov Cosmetics Caribbean Wedding Collection,nail_polish,14.99,2.0,13.12,1.87
365,moov,Moov Cosmetics Home Grown Canuck Collection,nail_polish,14.99,3.0,13.12,1.87
366,orly,Orly EPIX Flexible Sealcoat,nail_polish,13.49,5.0,13.12,0.37


## Output: 
Bronzer and Foundation have the most products above their category avg.
Notably, some highly-rated products carry only a modest premium over the avg.

## QUERY 11: Description Length and Price — Testing the "Description Premium" 
### What:  Bucket products by description length into tiers; compare avg price and
###        avg rating across tiers.
### Why:   Tests Hypothesis 2 — do brands with longer/more elaborate descriptions
###       charge more? A positive price gradient across description tiers would
###       support this claim.
### How:   CTE (style 2 subquery — WITH clause) creates the tier labels; the outer
###        query then aggregates. Using a CTE keeps the tier logic reusable and
###        separates classification from aggregation.


In [18]:
#We create a temporary table using WITH named desc_tiers. Then, we SELECT id, brand, product type, price, rating, description length, and word count 
#as well as feature engineers description categories called desc_tier using conditionals of description length = 0 being "no description", description length < 100 "Short (<100)", etc.
# all the way to very long (+600) filterd using WHERE to price larger than 0
#then, for the second part, we select this new desc_tier column as well as the count of all products, the average character count, average word count, the avg, min, and max price, and average rating
# we group by desc_tiers and ORDER BY desc tier.

q11 = """
WITH desc_tiers AS (
    SELECT
        id,
        brand,
        product_type,
        price,
        rating,
        description_length,
        word_count,
        CASE
            WHEN description_length = 0    THEN '0_No Description'
            WHEN description_length < 100  THEN '1_Short      (<100 chars)'
            WHEN description_length < 300  THEN '2_Medium     (100–299)'
            WHEN description_length < 600  THEN '3_Long       (300–599)'
            ELSE                                '4_Very Long  (600+)'
        END AS desc_tier
    FROM products
    WHERE price > 0
)
SELECT
    desc_tier,
    COUNT(*)                            AS product_count,
    ROUND(AVG(description_length), 0)   AS avg_char_count,
    ROUND(AVG(word_count), 0)           AS avg_word_count,
    ROUND(AVG(price), 2)                AS avg_price,
    ROUND(MIN(price), 2)                AS min_price,
    ROUND(MAX(price), 2)                AS max_price,
    ROUND(AVG(rating), 2)               AS avg_rating
FROM desc_tiers
GROUP BY desc_tier
ORDER BY desc_tier;
"""

print("=== Query 11: Description Length vs Price (CTE + GROUP BY) ===")
pd.read_sql(q11, conn)


=== Query 11: Description Length vs Price (CTE + GROUP BY) ===


,desc_tier,product_count,avg_char_count,avg_word_count,avg_price,min_price,max_price,avg_rating
0,0_No Description,23,0.0,0.0,15.10,4.50,39.0,NaN
1,1_Short (<100 chars),72,50.0,8.0,27.35,16.00,75.0,NaN
2,2_Medium (100–299),270,212.0,33.0,15.87,1.99,77.0,4.36
3,3_Long (300–599),259,420.0,63.0,13.86,1.99,45.0,4.31
4,4_Very Long (600+),253,1559.0,221.0,19.55,1.99,60.0,4.32


## Output: 
Products with very long descriptions (600+ chars) have a noticeably
higher avg price, providing preliminary support for the "description premium"
hypothesis. However, avg ratings do not clearly improve with description length.


## QUERY 12: Price Tier Cross-Tabulated with Green Claims 
### What:  For each combination of product_type and price tier, count products and
###        compute the % that carry green/ethical marketing claims.
### Why:   Tests whether green claims are concentrated in premium price tiers —
###        i.e., is "green" a luxury positioning strategy?
### How:   Derived table in FROM clause (subquery style 1 used in FROM) creates
###        price_tier per product id; explicit JOIN connects it back to the main table.

In [19]:
# Here we are creating the variable "q12", where the product table from our API and the price tiers are being selected. We are counting
# the number of values and assigning them to the column "product_count" and getting the average rating, rounding it to two decimal places
# and assigning it to the column "avg_rating"

# Under the column green_count we are counting and pattern matching how many product tags have the words "vegan", "organic", "natural", and 
# "cruelty". Next, we use the ROUND operator to get the percentage of these.

# We join this table back to our main table with price buckers of "budget", "mid-range", "premium", and "luxury", finally grouping by product 
# type and tier

q12 = """
SELECT
    p.product_type,
    tiers.price_tier,
    COUNT(*)                                                               AS product_count,
    ROUND(AVG(p.rating), 2)                                               AS avg_rating,
    SUM(CASE WHEN p.tags LIKE '%vegan%'
              OR p.tags LIKE '%organic%'
              OR p.tags LIKE '%natural%'
              OR p.tags LIKE '%cruelty%'  THEN 1 ELSE 0 END)              AS green_count,
    ROUND(
        100.0 * SUM(CASE WHEN p.tags LIKE '%vegan%'
                          OR p.tags LIKE '%organic%'
                          OR p.tags LIKE '%natural%'
                          OR p.tags LIKE '%cruelty%' THEN 1 ELSE 0 END)
        / COUNT(*), 1)                                                     AS pct_green
FROM products p
JOIN (
    SELECT
        id,
        CASE
            WHEN price < 10  THEN '1_Budget'
            WHEN price < 20  THEN '2_Mid-range'
            WHEN price < 35  THEN '3_Premium'
            ELSE             '4_Luxury'
        END AS price_tier
    FROM products
    WHERE price > 0
) tiers ON p.id = tiers.id
WHERE p.price > 0
  AND p.product_type IS NOT NULL
GROUP BY p.product_type, tiers.price_tier
ORDER BY p.product_type, tiers.price_tier;
"""

print("=== Query 12: Price Tier × Green Claims (Derived Table JOIN #2) ===")
pd.read_sql(q12, conn)

# We then execute the query get the following table:



=== Query 12: Price Tier × Green Claims (Derived Table JOIN #2) ===


,product_type,price_tier,product_count,avg_rating,green_count,pct_green
0,blush,1_Budget,20,4.23,2,10.0
1,blush,2_Mid-range,26,4.53,4,15.4
2,blush,3_Premium,23,4.37,4,17.4
3,blush,4_Luxury,3,NaN,0,0.0
4,bronzer,1_Budget,13,4.42,3,23.1
5,bronzer,2_Mid-range,19,4.63,2,10.5
6,bronzer,3_Premium,21,4.59,5,23.8
7,bronzer,4_Luxury,14,4.83,2,14.3
8,eyebrow,1_Budget,12,NaN,0,0.0
9,eyebrow,2_Mid-range,11,NaN,0,0.0


## Output: 
Green claims are NOT exclusively a luxury strategy — they appear across
all price tiers. Budget products actually have a notable green tag rate,
especially in lip products.

## QUERY 13: Brand-Level Green Positioning 
### What:  For each brand (with ≥5 products), show the share of green-tagged products
###        alongside that brand's avg price and avg rating per product_type.
### Why:   Identifies brands that systematically market themselves as green vs those
###        that do so selectively. High green% brands with low avg ratings may
###        signal greenwashing.
### How:   JOIN between the products table and a derived subquery (brand_stats) that
###        pre-aggregates green product counts per brand. This is JOIN #3.

In [20]:
# On this query, we JOIN our products data based on the subquery selecting brand, product count, and the conditional sum of brand green counts. Filtered by
# brand being not null and price >0, grouping by brand and hacing count(*) be larger or equal to 5.
# We SELECT product brand, product type, the count of all products, as well as the rounded averages of price and rating.
# We also selected the brand green count, brand total, and brand percentage from the brand_stats. 
#Then, we filter by price > 0 and product type being NOT NULL. Grouped by product brand and product type, ordered by brand percentage green (descending), and brand.

q13 = """
SELECT
    p.brand,
    p.product_type,
    COUNT(*)                                                                    AS products,
    ROUND(AVG(p.price), 2)                                                      AS avg_price,
    ROUND(AVG(p.rating), 2)                                                     AS avg_rating,
    brand_stats.brand_green_count,
    brand_stats.brand_total,
    ROUND(100.0 * brand_stats.brand_green_count
               / brand_stats.brand_total, 1)                                    AS brand_pct_green
FROM products p
JOIN (
    SELECT
        brand,
        COUNT(*)                                                                AS brand_total,
        SUM(CASE WHEN tags LIKE '%vegan%'
                  OR tags LIKE '%organic%'
                  OR tags LIKE '%natural%'
                  OR tags LIKE '%cruelty%' THEN 1 ELSE 0 END)                  AS brand_green_count
    FROM products
    WHERE brand IS NOT NULL
      AND price > 0
    GROUP BY brand
    HAVING COUNT(*) >= 5
) brand_stats ON p.brand = brand_stats.brand
WHERE p.price > 0
  AND p.product_type IS NOT NULL
GROUP BY p.brand, p.product_type
ORDER BY brand_pct_green DESC, p.brand;
"""

print("=== Query 13: Brand-Level Green Positioning (JOIN #3) ===")
pd.read_sql(q13, conn)


=== Query 13: Brand-Level Green Positioning (JOIN #3) ===


,brand,product_type,products,avg_price,avg_rating,brand_green_count,brand_total,brand_pct_green
0,pacifica,blush,1,28.00,3.6,13,13,100.0
1,pacifica,bronzer,1,60.00,4.9,13,13,100.0
2,pacifica,eyeliner,1,16.00,5.0,13,13,100.0
3,pacifica,eyeshadow,4,24.49,4.2,13,13,100.0
4,pacifica,lip_liner,1,16.00,NaN,13,13,100.0
...,...,...,...,...,...,...,...,...
170,wet n wild,eyeliner,5,5.69,4.5,0,12,0.0
171,wet n wild,eyeshadow,3,3.92,4.6,0,12,0.0
172,wet n wild,lipstick,2,2.99,4.6,0,12,0.0
173,wet n wild,mascara,1,3.49,5.0,0,12,0.0


## Output: 
- Some brands (e.g., e.l.f., physicians formula) have 100% green-tagged
- products while maintaining budget-to-mid price points — suggesting green
- positioning is viable across price tiers, not just luxury.


In [21]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)


In [22]:
import requests
import pandas as pd
import numpy as np

# copy the cell of the midterm project
BASE_URL = "http://makeup-api.herokuapp.com/api/v1/products.json"
df_raw = pd.DataFrame(requests.get(BASE_URL, timeout=30).json())
df_raw["price_num"] = pd.to_numeric(df_raw["price"], errors="coerce")
df_raw["rating"]    = pd.to_numeric(df_raw["rating"], errors="coerce")

analysis_cols = ["brand", "name", "product_type", "price_num", "rating"]
df_analysis = (
    df_raw[analysis_cols]
    .dropna(subset=["price_num", "rating"])
    .loc[df_raw["price_num"] > 0]
    .copy()
)


df_analysis["rating_tier"] = pd.cut(
    df_analysis["rating"], bins=[0, 4.0, 4.5, 5.0],
    labels=["Low", "Medium", "High"], include_lowest=True
)
type_avg = df_analysis.groupby("product_type")["rating"].mean()
df_analysis["type_avg_rating"] = df_analysis["product_type"].map(type_avg)
df_analysis["recommended"]     = (df_analysis["rating"] > df_analysis["type_avg_rating"]).astype(int)

print(df_analysis.shape)


(340, 8)


### Add description_length from df_raw (not in df_analysis) 

In [23]:
df_raw_desc = (
    df_raw[["brand", "name", "description"]]
    .assign(
        description_length = df_raw["description"].fillna("").str.len(),
        word_count         = df_raw["description"].fillna("").str.split().str.len()
    )
    .drop_duplicates(subset=["brand", "name"])
)

### Build clean modeling base (avoid duplicate columns from earlier merge) 

In [24]:

df_model = (
    df_analysis[["brand", "name", "product_type", "price_num",
                 "rating", "rating_tier", "type_avg_rating", "recommended"]]
    .copy()
    .merge(df_raw_desc[["brand", "name", "description_length", "word_count"]],
           on=["brand", "name"], how="left")
)
df_model[["description_length", "word_count"]] = (
    df_model[["description_length", "word_count"]].fillna(0)
)

print(f"Modeling dataset shape: {df_model.shape}")
print(f"\nTarget distribution — recommended:\n{df_model['recommended'].value_counts()}")
df_model.head()


Modeling dataset shape: (340, 10)

Target distribution — recommended:
recommended
1    197
0    143
Name: count, dtype: int64


,brand,name,product_type,price_num,rating,rating_tier,type_avg_rating,recommended,description_length,word_count
0,nyx,NYX Mosaic Powder Blush Paradise,bronzer,10.49,5.0,High,4.608333,1,344,56
1,annabelle,Annabelle Biggy Bronzer Haute Gold,bronzer,11.99,5.0,High,4.608333,1,130,20
2,physicians formula,Physicians Formula Super BB InstaReady Filter ...,bronzer,19.99,4.0,Low,4.608333,0,1287,158
3,maybelline,Maybelline Face Studio Master Hi-Light Light B...,bronzer,14.99,5.0,High,4.608333,1,367,52
4,physicians formula,Physicians Formula Bronze Booster Glow-Boostin...,bronzer,20.99,4.7,High,4.608333,1,1121,143
